# 실험4 — 레짐 맵: chunk size K x 실행 stride (TE off)

은지님 8/24 요청("ACT 대비 월등히 높은 세팅 찾기 / 세팅마다 ACT·BiMamba only·Carry only·BiMOS 4종")의 실행판.

## 왜 이 축인가
8/18 랩미팅 결론 — **BiMamba는 긴 stride, Carry는 짧은 stride에서 산다.** 그런데 그 근거표(슬라이드 93/94)는
libero/aloha 값이 섞여 stride 10·50·75 행이 두 벤치마크에 동일하게 들어가 있다. 검증 가능한 건
LIBERO K=100 stride=100 행뿐(ACT 18.8 / ACM2 25.3 / BiMamba 31.9 / BiMOS 25.6). **그래서 다시 뜬다.**

## 중요 제약 2가지 (코드 확인함)
1. **TE는 stride와 같이 못 쓴다.** `configuration_acm2.py`:
   `temporal_ensemble_coeff is not None and n_action_steps > 1` -> `NotImplementedError`.
   즉 TE => stride=1. 그래서 이 노트북은 **TE off 전용**이고, TE 축은 `exp3_*_te_ksweep` 이 담당한다.
2. **stride(`n_action_steps`)는 추론 파라미터 -> 재학습 불필요.** 기존 ckpt에 eval 플래그만 바꿔 던진다.
   K만 재학습이 필요하다.

## BiMamba only 의 교란 (반드시 읽을 것)
`MODEL_CONFIGS`에 carry 없는 BiMamba 태그가 **없다**. `bimamba` = `_CARRY_ON` + `use_chunk_pairs=True`.
8/18에 은지님이 "bimamba only를 전부 켜고 돌려서 다시 측정해야겠다"고 하신 그 문제가 코드에 그대로 남아 있다.
-> 이 노트북은 2단계로 간다.
   - **1단계(싼 것)**: 기존 ckpt에 `sscp_enabled=false` 만 걸어 eval. 라벨 = `bimamba_cpoff`.
     **학습은 chunk-pair로 된 상태**라 순수 BiMamba가 아니다. 표에 그대로 표기한다.
   - **2단계(비싼 것)**: 1단계에서 유망한 K만 `use_chunk_pairs=False` 로 **새로 학습**해 순수 BiMamba 확보.


## 0) 부팅 + 태그 등록

In [ ]:
import sys, json
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)
v23 = cf.v23

TASK   = 'libero_10'
K_LIST = [50, 100]                 # 기존 ckpt 있는 것부터. 확장 시 [20, 50, 100, 150]
SEEDS  = [0]                       # 경향 먼저 1 seed. 확정 표는 [0, 1]
N_EP   = 50                        # 은지님 지정. LIBERO-10 -> task당 50 = overall 500
GPUS   = v23.available_gpus()

_CARRY_ON, _CARRY_OFF = v23._CARRY_ON, v23._CARRY_OFF
STRIDE = lambda s: '--policy.n_action_steps=' + str(s)

def _tag(base, K):
    return base if K == 100 else base + '_k' + str(K)

# (라벨, 학습태그 base, eval 추가 플래그, 비고)
VARIANTS = [
    ('act',           'act',        [],            'pure ACT'),
    ('carry',         'acm2_carry', [_CARRY_ON],   'carry only (no BiMamba)'),
    ('bimamba_cpoff', 'bimamba',    [_CARRY_OFF],  'chunk-pair trained! not pure BiMamba'),
    ('bimos',         'bimamba',    [_CARRY_ON],   'BiMamba + carry'),
]

def strides_for(K):
    return [s for s in (1, 10, 25, 50, 75, 100) if s <= K]

def out_tag(label, K, s):
    return 'rm_' + label + '_k' + str(K) + '_s' + str(s)

# K!=100 태그는 MODEL_CONFIGS 에 없다 -> base 설정에서 K 만 갈아끼워 등록.
# (등록 안 하면 run_training_jobs -> make_train_cmd 에서 KeyError: 'acm2_carry_k50')
def reg_tag(base, K):
    t = _tag(base, K)
    if t not in v23.MODEL_CONFIGS:
        policy_type, lr, _K, extra, use_cp = v23.MODEL_CONFIGS[base]
        v23.MODEL_CONFIGS[t] = (policy_type, lr, K, list(extra), use_cp)
        v23.MODEL_LABELS.setdefault(t, v23.MODEL_LABELS.get(base, base) + ' (K=' + str(K) + ')')
    v23.MODEL_DIR_NAMES.setdefault(t, t)
    return t

for label, base, _f, _n in VARIANTS:
    for K in K_LIST:
        reg_tag(base, K)
        for s in strides_for(K):
            v23.MODEL_DIR_NAMES.setdefault(out_tag(label, K, s), out_tag(label, K, s))

print('train tags:', sorted({_tag(b, K) for _l, b, _f, _n in VARIANTS for K in K_LIST}))

total = sum(len(strides_for(K)) for K in K_LIST) * len(VARIANTS) * len(SEEDS)
print('K=', K_LIST, '| strides=', [strides_for(K) for K in K_LIST])
print('variants=', len(VARIANTS), '| seeds=', SEEDS, '| GPU=', GPUS)
print('-> eval total', total, 'runs (per-task', N_EP, 'ep, overall', N_EP * 10, 'ep)')


## 1) 사전 점검 — 어떤 ckpt가 이미 있나

**여기서 '없음' 으로 나오는 것만 학습하면 된다.** 다 있으면 학습 0, 전부 eval-only.

In [ ]:
missing = []
for label, base, _f, _n in VARIANTS:
    for K in K_LIST:
        for sd in SEEDS:
            t = _tag(base, K)
            try:
                cd = v23.best_ckpt_dir(t, sd, TASK, how=cf.CKPT_STEP)
                ok = cd is not None
            except Exception:
                ok = False
            print(('  OK   ' if ok else '  MISS ') + label.ljust(16) + '<- ' + t + '/seed' + str(sd))
            if not ok and (t, sd) not in missing:
                missing.append((t, sd))
print('')
if missing:
    print('need training:', missing)
else:
    print('*** all checkpoints present -> skip training, go straight to eval')


## 2) (필요시) 부족한 ckpt 학습

In [ ]:
train_jobs = [(t, sd, TASK) for (t, sd) in missing]
if train_jobs:
    print('training:', train_jobs)
    cf.run_training_jobs(train_jobs, GPUS, prefetch_task=TASK)
else:
    print('nothing to train')


## 3) eval — K x stride x 4변형 (재학습 X)

stride 는 `--policy.n_action_steps` 오버라이드로만 바뀐다. 유효 완료분은 스킵.

In [ ]:
_MINEP = 10 * N_EP // 2

def _has_valid(ot, sd):
    info = v23.eval_clean_dir(ot, sd, TASK) / 'eval_info.json'
    if not info.exists():
        return False
    try:
        ov = json.loads(info.read_text()).get('overall', {})
        return (ov.get('n_ep', ov.get('n_episodes')) or 0) >= _MINEP
    except Exception:
        return False

jobs = []
for K in K_LIST:
    for s in strides_for(K):
        for label, base, flags, _n in VARIANTS:
            for sd in SEEDS:
                ot = out_tag(label, K, s)
                if not _has_valid(ot, sd):
                    jobs.append((_tag(base, K), sd, flags + [STRIDE(s)], ot))
print('eval to run:', len(jobs), '/', total, '(completed skipped)')

ng = max(1, len(GPUS))
for i in range(0, len(jobs), ng):
    chunk = jobs[i:i + ng]
    labeled = []
    for g, (src, sd, flags, ot) in zip(GPUS, chunk):
        try:
            cmd = v23.make_eval_cmd(src, seed=sd, task=TASK, gpu_id=g, n_episodes=N_EP,
                                    select=cf.CKPT_STEP, extra_policy=flags,
                                    out_dir=v23.eval_clean_dir(ot, sd, TASK))
            labeled.append((ot + '/seed' + str(sd), cmd))
        except FileNotFoundError as e:
            print('  skip:', e)
    if labeled:
        print('')
        print('===== eval chunk ' + str(i // ng + 1) + ' (' + str(len(labeled)) + ' run) =====')
        v23.launch_cmds_live(labeled)


## 4) 레짐 맵 표 — 행=stride, 열=변형, 셀=SR

**읽는 법**
- `act` 열보다 높은 셀이 "ACT가 무너지는 지점". 그게 논문의 근거가 된다.
- `bimamba_cpoff` 가 긴 stride에서, `carry` 가 짧은 stride에서 이기면 8/18 가설 확정.
- `bimos` 가 둘 다한테 지면 **통합 모델은 폐기**하고 레짐 논문으로 간다.

In [ ]:
import pandas as pd

def _sr(ot, sd):
    info = v23.eval_clean_dir(ot, sd, TASK) / 'eval_info.json'
    if not info.exists():
        return None
    try:
        ov = json.loads(info.read_text()).get('overall', {})
        # pc_success 는 harness 전반에서 이미 percent 단위(show_eval_curve 가 %로 출력)
        return ov.get('pc_success')
    except Exception:
        return None

rows = []
for K in K_LIST:
    for s in strides_for(K):
        r = {'K': K, 'stride': s}
        for label, _b, _f, _n in VARIANTS:
            vals = [x for x in (_sr(out_tag(label, K, s), sd) for sd in SEEDS) if x is not None]
            r[label] = round(sum(vals) / len(vals), 1) if vals else None
            r[label + '_n'] = len(vals)
        rows.append(r)
df = pd.DataFrame(rows)

labels = [l for l, _b, _f, _n in VARIANTS]
df['best'] = df[labels].idxmax(axis=1)
df['best_minus_act'] = (df[labels].max(axis=1) - df['act']).round(1)
print(df[['K', 'stride'] + labels + ['best', 'best_minus_act']].to_string(index=False))

out = Path('exp4_regime_map.csv')
df.to_csv(out, index=False)
print('')
print('saved:', out.resolve())

win = df[(df['best'] != 'act') & (df['best_minus_act'] > 0)]
print('')
print('*** cells beating ACT:', len(win))
if len(win):
    print(win[['K', 'stride', 'act', 'best', 'best_minus_act']].to_string(index=False))
else:
    print('  none -> check TE axis (exp3_act_te_ksweep) before concluding')


## 5) 다음 단계 판단

| 4)의 결과 | 다음 |
|---|---|
| `bimamba_cpoff` 가 특정 (K, 긴 stride)에서 ACT 상회 | 그 K만 `use_chunk_pairs=False` 로 **순수 BiMamba 재학습** -> 교란 제거하고 확정. seed 늘리기 |
| `carry` 가 짧은 stride에서 ACT 상회 | carry 쪽으로 논문 축 이동. TE 축(`exp3_carry_te_ksweep`)과 합쳐 봐야 함 |
| 둘 다 되는데 `bimos` 만 안 됨 | **레짐 논문**. "두 메커니즘은 보완재가 아니라 대체재" 가 기여 |
| 아무것도 ACT를 못 이김 | TE 축(`exp3_act_te_ksweep`) 결과 먼저 확인 후 판단 |

**TE 축은 이 노트북이 아니라 `exp3_act_te_ksweep.ipynb` / `exp3_carry_te_ksweep.ipynb` 담당.**
특히 전자는 아직 한 번도 안 돌았다(outputs 비어 있음) — 표의 `act+TE` 열이 비어 있는 이유.
**그게 최우선.** BiMamba+TE 49.3 이 ACT+TE 를 이기는지가 논문 존폐를 가른다.